# L5: Automate Event Planning

In this lesson, you will learn more about Tasks.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:

!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [1]:
import sys
import os

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key, serper_api_key

import os

os.environ["OPENAI_MODEL_NAME"] = 'gpt-4-turbo'
os.environ["OPENAI_API_KEY"] = api_key
os.environ["SERPER_API_KEY"] = serper_api_key

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [3]:
from crewai import Agent, Crew, Task

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

## crewAI Tools

In [5]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(
/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/pydantic/_internal/_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)


## Creating Agents

In [6]:
# Agent 1: Venue Coordinator
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [8]:
# Agent 2: Logistics Manager
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipment"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [13]:
# Agent 3: Marketing and Communications Agent
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

## Creating Venue Pydantic Object

- Create a class `VenueDetails` using [Pydantic BaseModel](https://docs.pydantic.dev/latest/api/base_model/).
- Agents will populate this object with information about different venues by creating different instances of it.

In [14]:
from pydantic import BaseModel
# Define a Pydantic model for venue details 
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

## Creating Tasks
- By using `output_json`, you can specify the structure of the output you want.
- By using `output_file`, you can get your output in a file.
- By setting `human_input=True`, the task will ask for human feedback (whether you like the results or not) before finalising it.

In [15]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file="venue_details.json",  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator
)

- By setting `async_execution=True`, it means the task can run in parallel with the tasks which come after it.

In [40]:
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=True,
    async_execution=False,
    agent=logistics_manager
)

In [41]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    async_execution=True,
    output_file="marketing_report.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

## Creating the Crew

**Note**: Since you set `async_execution=True` for `logistics_task` and `marketing_task` tasks, now the order for them does not matter in the `tasks` list.

In [42]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[venue_coordinator, 
            logistics_manager, 
            marketing_communications_agent],
    
    tasks=[venue_task, 
           logistics_task, 
           marketing_task],
    
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [43]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall"
}

**Note 1**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

**Note 2**: 
- Since you set `human_input=True` for some tasks, the execution will ask for your input before it finishes running.
- When it asks for feedback, use your mouse pointer to first click in the text box before typing anything.

In [31]:
result = event_management_crew.kickoff(inputs=event_details)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 134e21d1-7e18-4be5-a59b-df23e6b25aa6                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Task: Find a venue in San Francisco that meets criteria for Tech Innovation Conference.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Thought: Thought: First, I need to find venues in San Francisco that can accommodate a Tech Innovation         │
│  Conference. I should use internet search to gather a list of potential venues, focusing on their capacity,     │
│  address, and booking availability.                                                                             │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"conference venues in San Francisco suitable for tech events\"}"                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  TOP 5 DIGITAL MARKETING STRATEGIES FOR 2023 - Contemporary Technology University                               │
│  Skip to content                                                                                                │
│  Skip to footer                                                                                                 │
│  Study Programs                                                                                                 │
│  Master of Science in Digital Marketing Master of Science in Computer Science Specialization in AI & Machine    │
│  Learning                                                                                                       │
│  Prospective Students                                                                                           │
│  Admission and Tuition Payment Options Fellowship Model Community English Proficiency                           │
│  About                                                                                                          │
│  About Contech Faculty Our Team 2024 Graduation Spotlight Scholarship Fund Policies Educational Approach        │
│  Academic Calendar Join Our Network                                                                             │
│  Events                                                                                                         │
│  Study Programs                                                                                                 │
│  Master of Science in Digital Marketing Master of Science in Computer Science Specialization in AI & Machine    │
│  Learning                                                                                                       │
│  Prospective Students                                                                                           │
│  Admission and Tuition Payment Options Fellowship Model Community English Proficiency                           │
│  About                                                                                                          │
│  About Contech Faculty Our Team 2024 Graduation Spotlight Scholarship Fund Policies Educational Approach        │
│  Academic Calendar Join Our Network                                                                             │
│  Events                                                                                                         │
│  Close                                                                                                          │
│  Study Programs                                                                                                 │
│  Master of Science in Digital Marketing Master of Science in Computer Science Specialization in AI & Machine    │
│  Learning                                                                                                       │
│  Prospective Students                                                                                           │
│  Admission and Tuition Payment Options Fellowship Model Community English Proficiency                           │
│  About                                                                                                          │
│  About Contech Faculty Our Team 2024 Graduation Spotlight Scholarship Fund Policies Educational Approach        │
│  Academic Calendar Join Our Network                                                                             │
│  Events                                                                                                         │
│  Have a Project?                                      

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing and Communications Agent                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Marketing and Communications Plan for the Tech Innovation Conference                                         │
│                                                                                                                 │
│  ## Digital Marketing Strategies                                                                                │
│                                                                                                                 │
│  ### 1. Personalization                                                                                         │
│  To effectively engage potential attendees, we will implement personalized marketing strategies. Using data     │
│  analytics, we will tailor our email campaigns and website content to meet the specific interests and needs of  │
│  our target audience. Personalized product recommendations and user-specific content can significantly          │
│  increase our conversion rates.                                                                                 │
│                                                                                                                 │
│  ### 2. Video Marketing                                                                                         │
│  Recognizing the growing dominance of video in online content consumption, we will produce high-quality promo   │
│  videos, testimonials, and live streams. These will be shared across platforms like YouTube, TikTok, and        │
│  Instagram to maximize reach and engagement.                                                                    │
│                                                                                                                 │
│  ### 3. Voice Search Optimization                                                                               │
│  As voice search becomes more popular, we will optimize our digital content to be voice search-friendly. We'll  │
│  incorporate long-tail keywords and structured data on our website to improve visibility when potential         │
│  attendees use devices like Google Home or Amazon Echo for searches related to tech conferences.                │
│                                                                                                                 │
│  ### 4. Influencer Marketing                                                                                    │
│  To tap into established audiences, we will partner with influencers within the tech industry. These            │
│  partnerships will help us reach a broader, yet targeted, audience. Influencers will share their genuine        │
│  experiences and the benefits of attending our conference, leveraging platforms like Instagram and TikTok.      │
│                                                                                                                 │
│  ### 5. Interactive Content                                                                                     │
│  We will increase audience engagement by creating interactive content such as polls, quizzes, and interactive   │
│  videos. This content will not only entertain potential attendees but also provide us with valuable insights    │
│  into their preferences and behavior.                                                                           │
│                                                        

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: cae25b0e-b14d-4897-a6b0-9a09a4dea04c                                                                     │
│  Agent: Marketing and Communications Agent                                                                      │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 134e21d1-7e18-4be5-a59b-df23e6b25aa6                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # Marketing and Communications Plan for the Tech Innovation Conference                           │
│                                                                                                                 │
│  ## Digital Marketing Strategies                                                                                │
│                                                                                                                 │
│  ### 1. Personalization                                                                                         │
│  To effectively engage potential attendees, we will implement personalized marketing strategies. Using data     │
│  analytics, we will tailor our email campaigns and website content to meet the specific interests and needs of  │
│  our target audience. Personalized product recommendations and user-specific content can significantly          │
│  increase our conversion rates.                                                                                 │
│                                                                                                                 │
│  ### 2. Video Marketing                                                                                         │
│  Recognizing the growing dominance of video in online content consumption, we will produce high-quality promo   │
│  videos, testimonials, and live streams. These will be shared across platforms like YouTube, TikTok, and        │
│  Instagram to maximize reach and engagement.                                                                    │
│                                                                                                                 │
│  ### 3. Voice Search Optimization                                                                               │
│  As voice search becomes more popular, we will optimize our digital content to be voice search-friendly. We'll  │
│  incorporate long-tail keywords and structured data on our website to improve visibility when potential         │
│  attendees use devices like Google Home or Amazon Echo for searches related to tech conferences.                │
│                                                                                                                 │
│  ### 4. Influencer Marketing                                                                                    │
│  To tap into established audiences, we will partner with influencers within the tech industry. These            │
│  partnerships will help us reach a broader, yet targeted, audience. Influencers will share their genuine        │
│  experiences and the benefits of attending our conference, leveraging platforms like Instagram and TikTok.      │
│                                                                                                                 │
│  ### 5. Interactive Content                                                                                     │
│  We will increase audience engagement by creating interactive content such as polls, quizzes, and interactive   │
│  videos. This content will not only entertain potential attendees but also provide us with valuable insights    │
│  into their preferences and behavior.                 

Display the generated venue_details.json file.

In [32]:
import json
from pprint import pprint

with open('venue_details.json', 'r') as f:
    data = json.load(f)

pprint(data)

{'address': '255 S Airport Blvd, South San Francisco, CA 94080',
 'booking_status': 'Contact for availability',
 'capacity': 20500,
 'name': 'South San Francisco Conference Center'}


- Display the generated `marketing_report.md` file.

**Note**: After `kickoff` execution has successfully ran, wait an extra 45 seconds for the `marketing_report.md` file to be generated. If you try to run the code below before the file has been generated, your output would look like:

```
marketing_report.md
```

If you see this output, wait some more and than try again.

In [33]:
from IPython.display import Markdown
Markdown(result.raw)

# Marketing and Communications Plan for the Tech Innovation Conference

## Digital Marketing Strategies

### 1. Personalization
To effectively engage potential attendees, we will implement personalized marketing strategies. Using data analytics, we will tailor our email campaigns and website content to meet the specific interests and needs of our target audience. Personalized product recommendations and user-specific content can significantly increase our conversion rates.

### 2. Video Marketing
Recognizing the growing dominance of video in online content consumption, we will produce high-quality promo videos, testimonials, and live streams. These will be shared across platforms like YouTube, TikTok, and Instagram to maximize reach and engagement.

### 3. Voice Search Optimization
As voice search becomes more popular, we will optimize our digital content to be voice search-friendly. We'll incorporate long-tail keywords and structured data on our website to improve visibility when potential attendees use devices like Google Home or Amazon Echo for searches related to tech conferences.

### 4. Influencer Marketing
To tap into established audiences, we will partner with influencers within the tech industry. These partnerships will help us reach a broader, yet targeted, audience. Influencers will share their genuine experiences and the benefits of attending our conference, leveraging platforms like Instagram and TikTok.

### 5. Interactive Content
We will increase audience engagement by creating interactive content such as polls, quizzes, and interactive videos. This content will not only entertain potential attendees but also provide us with valuable insights into their preferences and behavior.

## Implementation Timeline

- **Q1 2024:** Launch initial personalized email campaigns and begin video production.
- **Q2 2024:** Roll out voice search optimization and finalize influencer partnerships.
- **Q3 2024:** Start distributing interactive content and begin monitoring engagement and feedback.

## Expected Outcomes

- **Attendee Engagement:** With the strategies in place, we aim to significantly boost engagement on our digital platforms and expect a higher conversion rate of interest into actual event attendees.
- **Brand Awareness:** Through influencer partnerships and compelling content, we anticipate increased brand visibility and recognition in the tech industry.
- **Data Collection and Insights:** Interactive content and personalized experiences will provide us with valuable data, enabling us to further refine our marketing strategies.

By focusing on these cutting-edge digital marketing strategies, we aim to not only meet but exceed our goal of engaging at least 500 potential attendees for the Tech Innovation Conference.